# MIRROR: MIMIC-IV, hospital-wide cohort (part 2 of 2)

Trains the reported MIRROR configuration on the MIMIC-IV, hospital-wide cohort with seed(s)
789, 1024, and evaluates it on patients the model never sees during training.
Paper: *MIRROR: Multimodal Integration for Drug Recommendation from Electronic
Health Records*. Code: [https://github.com/Zied-Zaafrani/mirror-drug-rec](https://github.com/Zied-Zaafrani/mirror-drug-rec).

| | |
|---|---|
| Accelerator | GPU T4 x1 |
| Expected time | about 5.8 h (a session stops at 12 h; rerunning resumes) |
| Code | the public repository, attached as a dataset or cloned (internet on) |
| Data | the private processed dataset, one folder per cohort (`mimic4_hospital/records.pkl`, ...) |

**Steps.** Attach the two inputs, choose *GPU T4*, then *Run all*. The notebook
checks that the partitions are the reported ones and that notes and laboratory
values load, trains each seed, prints the results next to the paper's value,
shows what the model recommends drug class by drug class, and saves the result
files as its output.

**Data use.** MIMIC-III and MIMIC-IV are available from PhysioNet under a
credentialed data use agreement. The processed data stay private. This notebook
publishes aggregate results only: the default `PUBLIC = True` leaves out the
per-patient cards and the trained weights.

In [ ]:
# 1. Settings. The defaults reproduce the reported configuration.
COHORT = "mimic4_hospital"
SEEDS = [789, 1024]
SHOWCASE_SEED = None          # keep this seed's model for the showcase; None to skip
CHANNELS = []                  # [] is the full model; e.g. ["--no_notes"] for an ablation
CHANNELS_FOR_DEMO = [c for c in CHANNELS if c in ("--no_notes", "--no_labs", "--no_copy_head")]
PUBLIC = True                  # True: no patient cards and no weights in the output
PATIENT_CARDS = 8              # how many held-out admissions to show when PUBLIC is False
SMOKE_EPOCHS = None            # 1 for a quick check of the whole notebook

EXPECTED_SIZES = [45584, 11449, 11495]   # train, validation and test instances of the reported runs
REPORTED_JACCARD = 0.5461
REPOSITORY_URL = "https://github.com/Zied-Zaafrani/mirror-drug-rec"
LOCAL_INPUT = "../../data-release"         # only used outside Kaggle
LOCAL_WORKING = "./reproduce-output"       # only used outside Kaggle

## Code and environment

In [ ]:
# 2. The code: the attached copy of the repository, or a fresh clone.
import hashlib, os, platform, shutil, subprocess, sys, time
from pathlib import Path

INPUT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path(LOCAL_INPUT).resolve()
WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(LOCAL_WORKING).resolve()
WORKING.mkdir(parents=True, exist_ok=True)
REPOSITORY = WORKING / "mirror"

def is_mirror(folder: Path) -> bool:
    return (folder / "src" / "train.py").exists() and (folder / "src" / "model.py").exists()

if not REPOSITORY.exists():
    attached = [p.parent.parent for p in INPUT.rglob("config.yaml")
                if p.parent.name == "src" and is_mirror(p.parent.parent)]
    if attached:
        shutil.copytree(attached[0], REPOSITORY, ignore=shutil.ignore_patterns("results", ".git"))
        print("Code copied from the attached dataset:", attached[0])
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY)], check=True)
        print("Code cloned from", REPOSITORY_URL)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "scikit-learn",
                "pyyaml"], check=True)   # torch is already installed with GPU support
os.chdir(REPOSITORY)
sys.path.insert(0, str(REPOSITORY / "src"))

# Fingerprint of the exact source that runs here, so a reader can match it to a commit.
digest = hashlib.sha256()
for path in sorted((REPOSITORY / "src").rglob("*.py")):
    digest.update(path.read_bytes())
import torch
print(f"python {platform.python_version()} | torch {torch.__version__} | "
      f"CUDA {torch.cuda.is_available()} | source sha256 {digest.hexdigest()[:16]}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Data, and the checks that it is the reported cohort

In [ ]:
# 3. The data: the private dataset holding one folder per cohort.
from dataset import PrescriptionDataset, load_cohort_artifacts, select, split_patients

found = sorted(INPUT.rglob(f"{COHORT}/records.pkl"))
assert found, (f"No folder '{COHORT}' with records.pkl under {INPUT}. "
               "Attach the private processed dataset (see the first cell).")
COHORT_DIR = found[0].parent
PREPARED_DATA = COHORT_DIR.parent
RESULTS = WORKING / "results"

print("Cohort folder:", COHORT_DIR)
print("Files:", ", ".join(sorted(p.name for p in COHORT_DIR.iterdir())))

# The partitions must be the ones behind the reported numbers, and both extra
# channels must load. A run on the wrong split or without notes or labs stops here.
artifacts = load_cohort_artifacts(COHORT_DIR, use_notes=True, use_labs=True, expected_lab_dim=400)
split = split_patients(len(artifacts.records))
sizes = [len(PrescriptionDataset(select(artifacts.records, part), artifacts))
         for part in (split.train, split.validation, split.test)]
print(f"Prediction instances  train {sizes[0]:,}  validation {sizes[1]:,}  test {sizes[2]:,}")
assert sizes == EXPECTED_SIZES, f"Partitions {sizes} differ from the reported {EXPECTED_SIZES}"
print(f"{len(artifacts.records):,} patients, {artifacts.drug_count} drug classes. "
      "Partitions match the reported runs; notes and laboratory values loaded.")
del artifacts

## Training

In [ ]:
# 4. Train. One process per seed, so each starts with a clean GPU. A seed that
# already has a result file is skipped, so after a timeout the notebook can simply
# be run again.
WEIGHTS_DIR = Path("/tmp/mirror-weights") if PUBLIC else WORKING / "weights"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
LOGS = RESULTS / "logs"
LOGS.mkdir(parents=True, exist_ok=True)

def result_file(seed):
    return next(iter(RESULTS.glob(f"{COHORT}/*/result_*_seed{seed}.json")), None)

for seed in SEEDS:
    if result_file(seed):
        print(f"seed {seed}: already done, {result_file(seed).name}")
        continue
    command = [sys.executable, "-u", "src/train.py", "--cohort", COHORT, "--seed", str(seed),
               "--device", DEVICE, "--data_dir", str(PREPARED_DATA), "--results_dir", str(RESULTS),
               *CHANNELS]
    if SMOKE_EPOCHS:
        command += ["--epochs", str(SMOKE_EPOCHS)]
    if seed == SHOWCASE_SEED:
        command += ["--save_model", str(WEIGHTS_DIR / f"{COHORT}_seed{seed}.pt")]
    started = time.time()
    with open(LOGS / f"{COHORT}_seed{seed}.log", "w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace")
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        process.wait()
    assert process.returncode == 0, f"seed {seed} failed, see {LOGS}"
    print(f"seed {seed}: {(time.time() - started) / 3600:.2f} h")

## Results

In [ ]:
# 5. Results of the seeds run here, next to the value reported in the paper.
import json, statistics
subprocess.run([sys.executable, "src/report.py", str(RESULTS)], check=True)

runs = [json.loads(p.read_text()) for p in sorted(RESULTS.glob(f"{COHORT}/*/result_*.json"))]
values = [r["metrics"]["jaccard"] for r in runs]
if values:
    mean = statistics.fmean(values)
    spread = statistics.stdev(values) if len(values) > 1 else 0.0
    print(f"\nJaccard over {len(values)} seed(s) here: {mean:.4f} +/- {spread:.4f} "
          f"(paper, five seeds: {REPORTED_JACCARD:.4f}; difference {mean - REPORTED_JACCARD:+.4f})")

## What the model recommends, by drug name

In [ ]:
# 6. What the model recommends, drug class by drug class, on the test patients.
# Counts only; counts from 1 to 10 are hidden. Safe to publish with the results.
weights = WEIGHTS_DIR / f"{COHORT}_seed{SHOWCASE_SEED}.pt" if SHOWCASE_SEED else None
if weights and weights.exists():
    subprocess.run([sys.executable, "src/showcase.py", "--cohort", COHORT,
                    "--weights", str(weights), "--data_dir", str(PREPARED_DATA),
                    "--device", DEVICE, *CHANNELS_FOR_DEMO], check=True)
else:
    print("This notebook does not keep a model; the showcase is in the notebook that runs seed 42.")

In [ ]:
# 7. Held-out patients, one card each: the drugs the model recommended, marked
# right [+], wrong [x] or missed [-], by name.
# These are individual records from a credentialed database. The PhysioNet data
# use agreement does not allow showing them to people who have not signed it, so
# they print only when PUBLIC is False, in a private copy of this notebook.
if not PUBLIC and weights and weights.exists():
    subprocess.run([sys.executable, "src/showcase.py", "--cohort", COHORT,
                    "--weights", str(weights), "--data_dir", str(PREPARED_DATA),
                    "--device", DEVICE, *CHANNELS_FOR_DEMO,
                    "--patients", str(PATIENT_CARDS)], check=True)
elif PUBLIC:
    print("Patient cards are not printed in a public notebook (PhysioNet data use agreement).\n"
          "Set PUBLIC = False in a private copy to see them.")

## Output

In [ ]:
# 8. Keep the result files and training logs as this notebook's output.
import zipfile
archive = WORKING / f"results_{COHORT}_{'_'.join(map(str, SEEDS))}.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zipped:
    for path in sorted(RESULTS.rglob("*")):
        if path.is_file():
            zipped.write(path, path.relative_to(RESULTS))
print("Written", archive.name)
if PUBLIC:
    shutil.rmtree(Path("/tmp/mirror-weights"), ignore_errors=True)   # weights are not published